In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import lightgbm as lgb
from statsmodels.stats.proportion import proportions_ztest

np.random.seed(42)

df = pd.read_parquet("../../data/processed/pickup_features_v2.parquet")
clean = pd.read_parquet("../../data/processed/pickup_clean.parquet", columns=['order_id','courier_id','aoi_id'])
df = df.merge(clean, on='order_id', how='left').reset_index(drop=True)

model = lgb.Booster(model_file="../../models/pickup_disruption_lgbm.txt")
MODEL_FEATURES = ['hour_of_day','courier_orders_so_far','day_of_week','accept_distance_km',
                  'courier_late_rate','aoi_disruption_rate','city_hour_rate','courier_city_rate',
                  'courier_zone_familiarity','courier_tenure_days','courier_load_3h',
                  'mins_since_last_accept','velocity_target','courier_running_rate',
                  'zone_running_rate','orders_per_courier','city']
X = df[MODEL_FEATURES].copy()
for c in ['city','day_of_week','hour_of_day']:
    X[c] = X[c].astype('category')
df['risk_score'] = model.predict(X)

threshold = np.percentile(df['risk_score'], 90)
df['flagged_high_risk'] = (df['risk_score'] >= threshold).astype(int)

print(f"Total pickups: {len(df):,}")
print(f"High-risk flagged (top 10%): {df['flagged_high_risk'].sum():,}")
print(f"Disruption rate among FLAGGED: {df[df['flagged_high_risk']==1]['is_disrupted'].mean():.4f}")
print(f"Disruption rate among NOT flagged: {df[df['flagged_high_risk']==0]['is_disrupted'].mean():.4f}")

Total pickups: 6,064,908
High-risk flagged (top 10%): 606,491
Disruption rate among FLAGGED: 0.1194
Disruption rate among NOT flagged: 0.0069


In [ ]:
df['group'] = np.random.choice(['A_control', 'B_treatment'], size=len(df), p=[0.5, 0.5])

INTERVENTION_EFFECT = 0.30   

def simulate_outcome(row):
    if row['group'] == 'A_control':
        return row['is_disrupted']                   
    else: 
        if row['flagged_high_risk'] == 1 and row['is_disrupted'] == 1:
            return 0 if np.random.random() < INTERVENTION_EFFECT else 1
        else:
            return row['is_disrupted']               

df['sim_outcome'] = df['is_disrupted']  
mask = (df['group']=='B_treatment') & (df['flagged_high_risk']==1) & (df['is_disrupted']==1)
averted = np.random.random(mask.sum()) < INTERVENTION_EFFECT
df.loc[mask, 'sim_outcome'] = np.where(averted, 0, 1)

control_rate = df[df['group']=='A_control']['sim_outcome'].mean()
treatment_rate = df[df['group']=='B_treatment']['sim_outcome'].mean()
print(f"Control (A) disruption rate:   {control_rate:.4f}")
print(f"Treatment (B) disruption rate: {treatment_rate:.4f}")
print(f"Absolute reduction: {control_rate - treatment_rate:.4f}")
print(f"Relative reduction: {(control_rate - treatment_rate)/control_rate*100:.1f}%")

Control (A) disruption rate:   0.0182
Treatment (B) disruption rate: 0.0146
Absolute reduction: 0.0037
Relative reduction: 20.0%


In [ ]:
control = df[df['group']=='A_control']
treatment = df[df['group']=='B_treatment']

disruptions = [control['sim_outcome'].sum(), treatment['sim_outcome'].sum()] 
totals = [len(control), len(treatment)]

z_stat, p_value = proportions_ztest(disruptions, totals)

print(f"Control:   {disruptions[0]:,} disruptions out of {totals[0]:,} ({disruptions[0]/totals[0]*100:.2f}%)")
print(f"Treatment: {disruptions[1]:,} disruptions out of {totals[1]:,} ({disruptions[1]/totals[1]*100:.2f}%)")
print(f"\nZ-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.2e}")
print(f"\nSignificant at 0.05? {'YES — the reduction is real, not luck' if p_value < 0.05 else 'NO — could be random chance'}")

Control:   55,315 disruptions out of 3,032,755 (1.82%)
Treatment: 44,221 disruptions out of 3,032,153 (1.46%)

Z-statistic: 35.425
P-value: 7.15e-275

Significant at 0.05? YES — the reduction is real, not luck


In [ ]:
print("SENSITIVITY ANALYSIS — does the conclusion hold across intervention strengths?\n")
print(f"{'Effect':>8} {'Treat Rate':>12} {'Reduction':>11} {'P-value':>12} {'Significant':>12}")
print("-" * 60)

results = []
for effect in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    sim = df['is_disrupted'].copy()
    mask = (df['group']=='B_treatment') & (df['flagged_high_risk']==1) & (df['is_disrupted']==1)
    averted = np.random.random(mask.sum()) < effect
    sim[mask] = np.where(averted, 0, 1)

    control = sim[df['group']=='A_control']
    treatment = sim[df['group']=='B_treatment']
    x1, n1 = control.sum(), len(control)
    x2, n2 = treatment.sum(), len(treatment)
    p1, p2 = x1/n1, x2/n2
    p_pool = (x1+x2)/(n1+n2)
    se = np.sqrt(p_pool*(1-p_pool)*(1/n1+1/n2))
    z = (p1-p2)/se
    pval = 2*(1-stats.norm.cdf(abs(z)))
    reduction = (p1-p2)/p1*100
    sig = "YES" if pval < 0.05 else "no"
    print(f"{effect*100:>6.0f}% {p2*100:>11.2f}% {reduction:>10.1f}% {pval:>12.2e} {sig:>12}")
    results.append((effect, reduction, pval))

SENSITIVITY ANALYSIS — does the conclusion hold across intervention strengths?

  Effect   Treat Rate   Reduction      P-value  Significant
------------------------------------------------------------
     5%        1.75%        3.9%     5.60e-11          YES
    10%        1.70%        7.0%     0.00e+00          YES
    20%        1.57%       13.7%     0.00e+00          YES
    30%        1.45%       20.6%     0.00e+00          YES
    40%        1.33%       26.9%     0.00e+00          YES
    50%        1.21%       33.5%     0.00e+00          YES
